## Load Dataset

In [2]:
import kagglehub
path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")

Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.


In [3]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/pima-indians-diabetes-database/diabetes.csv


In [16]:
import pandas as pd
df = pd.read_csv('/kaggle/input/pima-indians-diabetes-database/diabetes.csv')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [6]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [7]:
df.sample(5)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
170,6,102,82,0,0,30.8,0.180,36,1
582,12,121,78,17,0,26.5,0.259,62,0
630,7,114,64,0,0,27.4,0.732,34,1
85,2,110,74,29,125,32.4,0.698,27,0
343,5,122,86,0,0,34.7,0.290,33,0


In [8]:
# finding missing values
df.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [9]:
# defining features and labels
X = df.drop(columns=["Outcome"])

y = df["Outcome"]

In [10]:
# checking the weightage
df["Outcome"].value_counts(normalize=True)*100

Outcome
0    65.104167
1    34.895833
Name: proportion, dtype: float64

In [11]:
# this ratio might create CLASS IMBALANCE.
# we need to stratify it
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
    )

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Scaling features (crucial for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# training the Logistic Regression model on the scaled data
lr_model = LogisticRegression()
lr_model.fit(X_train_scaled, y_train)

# Prediction
predictions = lr_model.predict(X_test_scaled)
print("Class Predictions: \n", predictions)

Class Predictions: 
 [1 0 0 0 0 0 0 1 0 1 0 1 0 0 0 0 1 0 1 0 0 1 0 1 1 0 1 0 0 0 0 0 0 1 1 0 0
 0 1 1 0 0 0 0 0 0 0 0 1 1 1 1 0 0 1 0 1 0 1 0 1 0 0 1 0 0 1 0 0 1 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 1 1 0 0 0 0 0 1 0 1 0 1 0 0
 1 0 0 0 0 0 0 1 0 1 0 0 1 0 1 1 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1
 1 0 0 0 1 0]


In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Calculate accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")

# Calculate precision
precision = precision_score(y_test, predictions)
print(f"Precision: {precision:.4f}")

# Calculate recall
recall = recall_score(y_test, predictions)
print(f"Recall: {recall:.4f}")

# Calculate F1-score
f1 = f1_score(y_test, predictions)
print(f"F1-Score: {f1:.4f}")

# Generate confusion matrix
conf_matrix = confusion_matrix(y_test, predictions)
print("\nConfusion Matrix:")
print(conf_matrix)

Accuracy: 0.7143
Precision: 0.6087
Recall: 0.5185
F1-Score: 0.5600

Confusion Matrix:
[[82 18]
 [26 28]]


In [15]:
import gradio as gr
import numpy as np

def predict_diabetes(pregnancies, glucose, bloodpressure, skinthickness, insulin, bmi, diabetespedigreefunction, age):
    # Create a numpy array from the input values
    input_data = np.array([
        pregnancies, glucose, bloodpressure, skinthickness,
        insulin, bmi, diabetespedigreefunction, age
    ]).reshape(1, -1)

    # Scale the input data using the trained scaler
    scaled_data = scaler.transform(input_data)

    # Make prediction using the trained logistic regression model
    prediction = lr_model.predict(scaled_data)

    # Map prediction to human-readable label
    return "Diabetic" if prediction[0] == 1 else "Not Diabetic"

# Build the UI using Blocks
with gr.Blocks() as interface:
    gr.Markdown("# Pima Indians Diabetes Prediction")
    gr.Markdown("Enter patient details to predict if they have diabetes.")

    with gr.Row():
        with gr.Column():
            pregnancies = gr.Number(label="Pregnancies")
            glucose = gr.Number(label="Glucose")
            bloodpressure = gr.Number(label="BloodPressure")
            skinthickness = gr.Number(label="SkinThickness")
        with gr.Column():
            insulin = gr.Number(label="Insulin")
            bmi = gr.Number(label="BMI")
            diabetespedigreefunction = gr.Number(label="DiabetesPedigreeFunction")
            age = gr.Number(label="Age")

    submit_btn = gr.Button("Predict")
    output = gr.Label(label="Prediction")

    # Connect the button to the prediction function
    submit_btn.click(
        fn=predict_diabetes,
        inputs=[pregnancies, glucose, bloodpressure, skinthickness, insulin, bmi, diabetespedigreefunction, age],
        outputs=output
    )

# Launch the interface
interface.launch(inline=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1366085576f5060230.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
